# Transform Sprints Data
1. Read bronze `sprints` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`constructorId` → `constructor_id`, `driverId` → `driver_id`, `raceName` → `race_name`, `positionText` → `finish_position_text`)
1. Rename columns to make them more meaningful (`date` → `race_date`, `grid` → `grid_position`, `laps` → `completed_laps`, `number` → `car_number`, `position` → `finish_position`)
1. Filter out rows where `season`, `round`, `custructor_id` or `driver_id` is null (business key validation)
1. Remove duplicate records
1. Transform values of column `race_name` to Title Case
1. Write the transformed data to silver `sprints` table

In [0]:
from pyspark.sql import functions as f

Step 1 - Loading the env config from the common config

In [0]:
%run ../00-common-Config/01-environment-variable

In [0]:
%run ../00-common-Config/02-helper-function

In [0]:
bronze_table=f"{catalog_name}.{bronze_schema}.sprints"
silver_table=f"{catalog_name}.{silver_schema}.sprints"

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
sprints_df=(
     spark.read.table(bronze_table)
     .filter(f.col("batch_id")==v_batch_id)
     .select(
         "constructorId",
         "date",
         "driverId",
         "grid",
         "laps",
         "number",
         "points",
         "position",
         "positionText",
         "raceName",
         "round",
         "season",
         "status",
         "ingestion_timestamp",
         "source_file",
         "batch_id"
     )
     .withColumnsRenamed(
        {
            "constructorId": "constructor_id",
            "driverId": "driver_id",
            "raceName":"race_name",
            "positionText":"finish_position_text",
            "date":"race_date",
            "grid":"grid_position",
            "laps":"compeleted_laps",
            "number":"car_number",
            "position":"finish_position"
        }
    )
)

In [0]:
sprints_valid_df=(
    sprints_df
    .filter(
    f.col("season").isNotNull() &
    f.col("round").isNotNull() &
    f.col("constructor_id").isNotNull() &
    f.col("driver_id").isNotNull()
)
.dropDuplicates(["season","round","constructor_id","driver_id"])
)

In [0]:
sprints_final_df=(
    sprints_valid_df
    .withColumn("race_name",f.initcap(f.col("race_name")))
)

In [0]:
# (
#     sprints_final_df
#     .write
#     .mode("overwrite")
#     .format("delta")
#     .saveAsTable(silver_table)
# )



write_to_silver(
    input_df=sprints_final_df,
    target_table=silver_table,
    merge_condition="""
        t.season = s.season AND 
        t.round = s.round AND 
        t.constructor_id = s.constructor_id AND 
        t.driver_id = s.driver_id
    """,
    columns_to_update=[
        "race_name",
        "race_date",
        "grid_position",
        "compeleted_laps",   # ✅ matches schema
        "car_number",
        "points",
        "finish_position",   # ✅ correct
        "finish_position_text",  # ✅ correct
        "status",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
display(spark.table(silver_table))